# Partie 1 — SparkSession connectée au Master standalone

Le driver tourne dans ce conteneur Jupyter ; les executors sont lancés par les deux Workers (`spark-worker-1`, `spark-worker-2`).
Spark UI de l'application : http://localhost:4040 — Master UI : http://localhost:8080

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("SahlPay-Partie1-Notebook")
         .master("spark://spark-master:7077")
         .getOrCreate())
sc = spark.sparkContext

print("Version Spark      :", spark.version)
print("Master             :", sc.master)
print("appName            :", sc.appName)
print("defaultParallelism :", sc.defaultParallelism)

In [ ]:
# Les tâches s'exécutent bien sur les executors des Workers (et non dans le driver)
import socket
sorted(sc.parallelize(range(8), 8).map(lambda _: socket.gethostname()).distinct().collect())

In [ ]:
# Accès au bucket MinIO "lake" via S3A (configuration dans spark-defaults.conf)
path = spark._jvm.org.apache.hadoop.fs.Path("s3a://lake/")
fs = path.getFileSystem(sc._jsc.hadoopConfiguration())
[s.getPath().getName() for s in fs.listStatus(path)]

In [ ]:
spark.stop()